# Prepare the FFHQ-Wrinkle dataset on Google Colab

Builds the dataset zip that `colab_train.ipynb` expects (`ffhq_wrinkle_data.zip` on your Drive),
**without downloading anything to your computer**. Run it **once**; afterwards only use `colab_train.ipynb`.

It produces, for the 1,000 labelled faces:

| Folder | Comes from |
|---|---|
| `manual_wrinkle_masks/` | `manual-wrinkle-masks.zip` (FFHQ-Wrinkle) |
| `weak_wrinkle_masks/` | the matching 1,000 files from `weak-wrinkle-masks.zip` (the other 49,000 are skipped) |
| `masked_face_images/` | FFHQ `images1024x1024` photos, with everything except the face blacked out using `face-parsed-labels.zip` |

**Before you start (in your browser, once):**
1. Open the FFHQ-Wrinkle Drive folder (linked from <https://github.com/labhai/ffhq-wrinkle-dataset>) →
   click the folder name `ffhq-wrinkle-dataset` → *Organize → Add shortcut* → **My Drive**.
2. Open the FFHQ Drive folder (linked from <https://github.com/NVlabs/ffhq-dataset>) → open `images1024x1024` →
   *Organize → Add shortcut* → **My Drive**.

Shortcuts use none of your Drive storage. Only the final zip (~1–2 GB) does.

A CPU runtime is enough for this notebook; no GPU needed. Check the dataset licenses
(FFHQ-Wrinkle: CC BY-NC-SA 4.0; FFHQ: see its README) before use.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Settings

Change the two shortcut paths only if your shortcuts have different names or live in a subfolder.

In [ ]:
import os, glob, shutil, zipfile

WRINKLE_DIR = '/content/drive/MyDrive/ffhq-wrinkle-dataset'   # shortcut 1
FFHQ_DIR    = '/content/drive/MyDrive/images1024x1024'        # shortcut 2
OUT_ZIP     = '/content/drive/MyDrive/ffhq_wrinkle_data.zip'  # must match DATA_ZIP in colab_train.ipynb

WORK = '/content/prep'   # scratch space (local disk)
OUT  = '/content/data'   # final dataset folder (same as DATA_ROOT in colab_train.ipynb)

ok = True
for z in ('manual-wrinkle-masks.zip', 'weak-wrinkle-masks.zip', 'face-parsed-labels.zip'):
    found = os.path.isfile(f'{WRINKLE_DIR}/{z}')
    ok &= found
    print(('OK      ' if found else 'MISSING ') + f'{WRINKLE_DIR}/{z}')
found = os.path.isdir(f'{FFHQ_DIR}/00000')
ok &= found
print(('OK      ' if found else 'MISSING ') + f'{FFHQ_DIR}/00000/')
if not ok:
    print('\nFix the shortcuts (see the top of this notebook). Zips on My Drive:')
    print(glob.glob('/content/drive/MyDrive/*'))

## Step 3 — Helpers

In [ ]:
def stem(path):
    return os.path.splitext(os.path.basename(path))[0]

def extract(zip_name, dest, exts=('.png',), keep=None):
    """Extract files with the given extensions from a zip on Drive, flattened into `dest`.
    If `keep` is given, only files whose name (without extension) is in `keep` are extracted."""
    os.makedirs(dest, exist_ok=True)
    n = 0
    with zipfile.ZipFile(f'{WRINKLE_DIR}/{zip_name}') as z:
        for info in z.infolist():
            name = info.filename
            if info.is_dir() or not name.lower().endswith(exts) or os.path.basename(name).startswith('._'):
                continue
            if keep is not None and stem(name) not in keep:
                continue
            target = os.path.join(dest, os.path.basename(name))
            if not os.path.exists(target):
                with z.open(info) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
            n += 1
    return n

## Step 4 — Manual wrinkle masks (defines the 1,000 labelled IDs)

In [ ]:
n = extract('manual-wrinkle-masks.zip', f'{OUT}/manual_wrinkle_masks')
IDS = sorted(stem(p) for p in glob.glob(f'{OUT}/manual_wrinkle_masks/*.png'))
print(f'{n} manual masks -> {len(IDS)} labelled IDs, e.g. {IDS[:5]}')

## Step 5 — Weak wrinkle masks (only the 1,000 that match)

Reads the 6.9 GB zip straight from Drive but extracts only the needed files. Takes a few minutes.

In [ ]:
n = extract('weak-wrinkle-masks.zip', f'{OUT}/weak_wrinkle_masks', keep=set(IDS))
have = {stem(p) for p in glob.glob(f'{OUT}/weak_wrinkle_masks/*.png')}
print(f'{len(have)} weak masks | missing: {sorted(set(IDS) - have)[:10]}')

## Step 6 — FFHQ face photos (only the 1,000 that match)

FFHQ stores photos in folders of 1,000 (`00000/`, `01000/`, ...). Safe to rerun: already-copied photos are skipped.
If you see *quota exceeded* / I/O errors, Google is rate-limiting the shared FFHQ folder: wait a few hours and rerun this cell.

In [ ]:
from tqdm.auto import tqdm

FACES = f'{WORK}/face_images'
os.makedirs(FACES, exist_ok=True)
failed = []
for i in tqdm(IDS):
    dst = f'{FACES}/{i}.png'
    if os.path.exists(dst):
        continue
    src = f'{FFHQ_DIR}/{int(i) // 1000 * 1000:05d}/{i}.png'
    try:
        shutil.copy(src, dst + '.part')
        os.replace(dst + '.part', dst)
    except OSError as e:
        failed.append((i, str(e)))
print(f'{len(os.listdir(FACES))} photos copied | failed: {len(failed)}')
for i, err in failed[:5]:
    print(' ', i, err)

## Step 7 — Mask the faces

Same procedure as `face_masking.py` in the FFHQ-Wrinkle repo: each 512×512 face-parsing label map is upscaled 2×
(nearest neighbour) and every pixel whose label is not **1** or **10** is set to black.

In [ ]:
import numpy as np
from PIL import Image

PARSED = f'{WORK}/face_parsed_labels'
extract('face-parsed-labels.zip', PARSED, exts=('.npy',))

MASKED = f'{OUT}/masked_face_images'
os.makedirs(MASKED, exist_ok=True)
skipped = []
for i in tqdm(IDS):
    face, label = f'{FACES}/{i}.png', f'{PARSED}/{i}.npy'
    if not (os.path.exists(face) and os.path.exists(label)):
        skipped.append(i)
        continue
    img = np.array(Image.open(face).convert('RGB'))
    lab = np.load(label)
    lab = lab.repeat(img.shape[0] // lab.shape[0], axis=0).repeat(img.shape[1] // lab.shape[1], axis=1)
    img[~((lab == 1) | (lab == 10))] = 0
    Image.fromarray(img).save(f'{MASKED}/{i}.png')
print(f'{len(os.listdir(MASKED))} masked faces | skipped (photo or label missing): {len(skipped)} {skipped[:10]}')

## Step 8 — Check, then save the zip to Drive

Every folder must have the same 1,000 IDs at 1024×1024. The zip is written only if the check passes.

In [ ]:
folders = ('masked_face_images', 'weak_wrinkle_masks', 'manual_wrinkle_masks')
sets = {d: {stem(p) for p in glob.glob(f'{OUT}/{d}/*.png')} for d in folders}
problems = []
for d, s in sets.items():
    print(f'{d:22s} {len(s)} files')
    if s != set(IDS):
        problems.append(f'{d}: missing {sorted(set(IDS) - s)[:5]}')
    with Image.open(f'{OUT}/{d}/{IDS[0]}.png') as im:
        print(f'{"":22s} sample {IDS[0]}: size={im.size} mode={im.mode}')
        if im.size != (1024, 1024):
            problems.append(f'{d}: size {im.size}')

if problems:
    print('\nNOT saved. Fix these first (usually: rerun Step 6, then Step 7):')
    print('\n'.join(problems))
else:
    !cd {OUT} && rm -f /content/dataset.zip && zip -qr /content/dataset.zip {' '.join(folders)}
    shutil.copy('/content/dataset.zip', OUT_ZIP)
    print(f'\nSaved {OUT_ZIP} ({os.path.getsize(OUT_ZIP) / 2**30:.2f} GB).')
    print('Now open colab_train.ipynb; its Step 4 will use this zip.')

## (Optional) Preview one sample

In [ ]:
import matplotlib.pyplot as plt
i = IDS[0]
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for a, d in zip(ax, folders):
    a.imshow(Image.open(f'{OUT}/{d}/{i}.png'), cmap='gray')
    a.set_title(f'{d}/{i}.png')
    a.axis('off')
plt.show()